# KG Construction — evaluation and annotation

This notebook uses the completed raw JSONL produced by the extraction notebook. It performs automatic evaluation, creates the manual-annotation package, and computes the final manual metrics after the CSV files have been labeled.

For the frozen experiment, P runs in `quote_only` mode; this notebook does not require a GPU or Ollama.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Settings

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/DKR-KG-Construction/KG-construction.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/KG-construction')
DRIVE_ROOT = Path('/content/drive/MyDrive/DKR_KG_Construction')

SPLIT = 'test'
RAW_PATH = DRIVE_ROOT / 'outputs' / 'raw' / f'{SPLIT}.jsonl'
AUTO_RESULTS_DIR = DRIVE_ROOT / 'results' / SPLIT
SAMPLE_DIR = DRIVE_ROOT / 'annotation' / 'proposal_test'
MANUAL_RESULTS_DIR = DRIVE_ROOT / 'results'

AUTO_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MANUAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Raw cache:', RAW_PATH)
print('Automatic results:', AUTO_RESULTS_DIR)
print('Annotation package:', SAMPLE_DIR)

## 2. Repository, dependencies, and dataset

TEST evaluation verifies the freeze manifest and checksums again, so the dataset must be present even when the raw cache is already complete.

In [ ]:
import os
import subprocess
import sys

def run(command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)])
else:
    run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH])
    run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH])
    run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])

run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements.lock')])
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
run(['bash', 'run.sh', 'data'], cwd=REPO_DIR)

if not RAW_PATH.is_file():
    raise FileNotFoundError(
        f'Final TEST extraction was not found: {RAW_PATH}. '
        'Run 03_colab_extract.ipynb before this notebook. '
        'For TRAIN/DEV configuration search, use notebooks 01 and 02 instead.'
    )

## 3. Check raw-extraction completeness

Frozen TEST evaluation requires all 500 unique document IDs and a matching extractor fingerprint. The evaluation command performs the full fingerprint validation again.

In [ ]:
sys.path.insert(0, str(REPO_DIR / 'src'))
from kgrepair.extract import load_raw

raw = load_raw(RAW_PATH)
normal = sum(r['parsed'] is not None and r.get('parse_error') is None for r in raw.values())
partial = sum(r.get('parse_error') == 'partial_json_recovered' for r in raw.values())
failed = sum(r['parsed'] is None for r in raw.values())
status = {'unique_documents': len(raw), 'normal': normal, 'partial': partial, 'failed': failed}
print(status)
if SPLIT == 'test' and len(raw) != 500:
    raise ValueError('TEST cache is incomplete. Resume extraction in notebook 01.')

## 4. Automatic evaluation

All eight C/S/P combinations will be evaluated for sample sizes of 25, 50, 100, and 200 documents across five seeds. Results are saved to Drive.

In [ ]:
evaluate_command = [
    sys.executable, '-m', 'kgrepair',
    '--config', str(REPO_DIR / 'config.yaml'),
    'evaluate', '--raw', str(RAW_PATH), '--split', SPLIT,
    '--no-judge', '--out-dir', str(AUTO_RESULTS_DIR),
]
run(evaluate_command, cwd=REPO_DIR, env=os.environ.copy())

In [ ]:
import pandas as pd

summary_path = AUTO_RESULTS_DIR / 'auto_summary.csv'
auto_summary = pd.read_csv(summary_path)
display(auto_summary)

## 5. Create the manual-annotation package

Run this step only once. The script deliberately refuses to overwrite existing `annotator1.csv`, `annotator2.csv`, and `key.json` files.

- `annotator1.csv`: the complete primary-annotator sample.
- `annotator2.csv`: the independent 20% overlap sample.
- `docs.txt`: full source documents used to verify facts.
- `key.json`: hidden stage mapping; do not share it with annotators before labeling is complete.

In [ ]:
sample_files = [SAMPLE_DIR / 'annotator1.csv', SAMPLE_DIR / 'annotator2.csv', SAMPLE_DIR / 'key.json']
if any(path.exists() for path in sample_files):
    print('Annotation package already exists; generation skipped:')
    for path in sample_files:
        print(' ', path, 'exists=' + str(path.exists()))
else:
    sample_command = [
        sys.executable, '-m', 'kgrepair',
        '--config', str(REPO_DIR / 'config.yaml'),
        'sample-proposal-annotation', '--raw', str(RAW_PATH),
        '--out-dir', str(SAMPLE_DIR), '--no-judge',
    ]
    run(sample_command, cwd=REPO_DIR, env=os.environ.copy())

## 6. Manual work between runs

Stop here and independently fill only the `label` and `notes` columns in both CSV files. Valid labels are `correct`, `incorrect`, and `uncertain`. Do not discuss overlap items until both independent versions have been saved.

After completing the files, return to the following cells.

In [ ]:
import csv

VALID_LABELS = {'correct', 'incorrect', 'uncertain'}

def label_status(path):
    with open(path, newline='', encoding='utf-8-sig') as stream:
        rows = list(csv.DictReader(stream))
    labels = [(row.get('label') or '').strip().lower() for row in rows]
    invalid = [label for label in labels if label and label not in VALID_LABELS]
    return {'rows': len(rows), 'filled': sum(bool(x) for x in labels), 'invalid': invalid}

for name in ('annotator1.csv', 'annotator2.csv'):
    path = SAMPLE_DIR / name
    print(name, label_status(path) if path.exists() else 'MISSING')

## 7. Final scoring

This cell runs scoring only when both assigned tables are completely and validly labeled. The result is copied from the repository's local `results/` directory to Google Drive.

In [ ]:
import shutil

a1 = SAMPLE_DIR / 'annotator1.csv'
a2 = SAMPLE_DIR / 'annotator2.csv'
statuses = [label_status(a1), label_status(a2)] if a1.exists() and a2.exists() else []
ready = bool(statuses) and all(s['rows'] == s['filled'] and not s['invalid'] for s in statuses)

if not ready:
    print('Scoring skipped: complete and validate both annotation CSV files first.')
else:
    score_command = [
        sys.executable, '-m', 'kgrepair',
        '--config', str(REPO_DIR / 'config.yaml'),
        'score-proposal-annotation',
        '--sample-dir', str(SAMPLE_DIR),
        '--labels', str(a1), '--labels2', str(a2),
    ]
    run(score_command, cwd=REPO_DIR, env=os.environ.copy())
    local_summary = REPO_DIR / 'results' / 'proposal_manual_summary.csv'
    drive_summary = MANUAL_RESULTS_DIR / 'proposal_manual_summary.csv'
    shutil.copy2(local_summary, drive_summary)
    print('Saved:', drive_summary)

In [ ]:
final_summary = MANUAL_RESULTS_DIR / 'proposal_manual_summary.csv'
if final_summary.exists():
    display(pd.read_csv(final_summary))
else:
    print('Final manual summary is not available yet.')

## Final artifacts

After the complete workflow, Drive should contain:

```text
DKR_KG_Construction/
├── outputs/raw/test.jsonl
├── results/test/auto_metrics.csv
├── results/test/auto_summary.csv
├── results/test/per_doc.csv
├── results/test/rejections.jsonl
├── results/test/run_metadata.json
├── results/proposal_manual_summary.csv
└── annotation/proposal_test/
    ├── annotator1.csv
    ├── annotator2.csv
    ├── docs.txt
    └── key.json
```